# Neurogenomic Regional Vulnerability

Do PD/AD/HD risk genes get expressed most in the exact brain region each disease destroys, and does that actually hold up statistically or is it just a nice-looking plot?

Run these top to bottom. Step 2 (AHBA download) is the slow part (first run downloads a few GB, cached after that) and needs to run twice - once to cache the donor data, then again after `build_sn_atlas.py` builds the corrected substantia-nigra region from that cached data.

Full writeup of the method, why each choice was made, and what the results actually mean is in the repo README and `report/report_draft.md`.

## setup

In [ ]:
!pip install -q abagen brainsmash nibabel "pandas>=1.5,<1.6" numpy scipy statsmodels matplotlib seaborn requests

# clone the repo if we're in colab and don't have src/ + data/gene_lists yet
import os
if not os.path.exists('src'):
    !git clone https://github.com/<your-username>/<your-repo>.git repo
    %cd repo

import sys
sys.path.insert(0, 'src')

## gene lists + expression
gene lists are already sitting in `data/gene_lists/` (see `fetch_gwas_genes.py` for how they were picked). this pulls AHBA expression for those genes across the DK atlas - first pass uses the plain atlas, since `build_sn_atlas.py` needs the donor data cached before it can find real substantia-nigra sample coordinates.

In [ ]:
!python src/expression_pipeline.py

## fix the substantia nigra region
the DK atlas's plain "brainstem" label failed the validation checkpoint badly (SNCA came out almost dead last). this carves substantia nigra out of it using real AHBA tissue-sample coordinates, then reruns the expression pull so it picks up the new atlas automatically.

In [ ]:
!python src/build_sn_atlas.py
!python src/expression_pipeline.py

## validation checkpoint
SNCA still doesn't rank high in substantia nigra even after the fix (rank 38 of 43) - turned out to be a more interesting finding than a simple pass/fail, see the report for why. The full 9-gene PD average does show up enriched there, which is what regional_comparison.py actually tests.

In [ ]:
!python src/validation_checkpoint.py

## regional comparison
target region vs. everywhere else, per disease.

In [ ]:
!python src/regional_comparison.py

## the actual significance test
BrainSMASH spatial nulls (1000 surrogates), FDR corrected across the three diseases.

In [ ]:
!python src/permutation_test.py

## figures

In [ ]:
!python src/visualize.py

from IPython.display import Image, display
for disease in ['parkinsons', 'alzheimers', 'huntingtons']:
    display(Image(f'results/figures/{disease}_regional_heatmap.png'))
display(Image('results/figures/results_summary.png'))

## results

Ran on real data: Parkinson's holds up after spatial-null correction (FDR p=0.015), Alzheimer's looks good raw but doesn't survive the spatial null (FDR p=0.22), Huntington's goes the opposite direction from the hypothesis. Full discussion of what I think this means in `report/report_draft.md`.